# Data Analysis Agent (Local Notebook)

Run this notebook locally in Jupyter or VS Code to launch the Web UI without Google Colab.

## Quick Start (Local)
1. Open this notebook locally.
2. Run Cell 2 (setup helpers).
3. Run Cell 3 (start server + clickable localhost URL).
4. Open the printed URL, upload CSV, type a question, and click Analyze Dataset.
5. (Optional) Run Cell 4 to stop the server.

In [1]:
import json
import os
import sys
import subprocess
import time
from getpass import getpass
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError

from IPython.display import HTML, display

PORT = 8000
HOST = "127.0.0.1"
LOCAL_HEALTH_URL = f"http://{HOST}:{PORT}"

def _is_project_root(path: Path) -> bool:
    return (path / "app.py").exists() and (path / "static").exists()

def _scan_for_project_root(base: Path, max_depth: int = 5) -> Path | None:
    if not base.exists() or not base.is_dir():
        return None

    base_depth = len(base.parts)
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        current_depth = len(root_path.parts) - base_depth
        if current_depth > max_depth:
            dirs[:] = []
            continue

        if "app.py" in files and (root_path / "static").exists():
            return root_path

    return None

def _find_project_root(start: Path) -> Path:
    # 1) Explicit override via env var.
    hint = os.getenv("PROJECT_ROOT_HINT", "").strip()
    if hint:
        hint_path = Path(hint).expanduser()
        if _is_project_root(hint_path):
            return hint_path

    # 2) Check cwd and all parents first.
    for candidate in [start, *start.parents]:
        if _is_project_root(candidate):
            return candidate

    # 3) Search common local locations.
    common_bases = [Path.cwd(), Path.home(), Path("/tmp")]
    for base in common_bases:
        found = _scan_for_project_root(base, max_depth=6)
        if found is not None:
            return found

    raise FileNotFoundError(
        "Could not locate project root containing app.py and static/.\n"
        "Set PROJECT_ROOT_HINT to your project folder before running this cell.\n"
        "Example: set PROJECT_ROOT_HINT=/path/to/CS539_Project in your environment."
    )

def _wait_for_server(url: str, timeout_seconds: int = 30) -> bool:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            with urlopen(f"{url}/health", timeout=2) as response:
                if response.status == 200:
                    return True
        except URLError:
            pass
        time.sleep(1)
    return False

def _get_health(url: str):
    try:
        with urlopen(f"{url}/health", timeout=3) as response:
            if response.status == 200:
                return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None
    return None

PROJECT_ROOT = _find_project_root(Path.cwd())
SERVER_PROCESS = None

print(f"Project root: {PROJECT_ROOT}")
print("Runtime: Local Jupyter/VS Code")
print(f"Server bind host: {HOST}:{PORT}")
print(f"Local health URL: {LOCAL_HEALTH_URL}")

Project root: /Users/jasonhu/Documents/GitHub/CS539_Project/CS539_Project
Runtime: Local Jupyter/VS Code
Server bind host: 127.0.0.1:8000
Local health URL: http://127.0.0.1:8000


In [2]:
# Start Web UI server (Local)
is_running = SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None

if is_running:
    health = _get_health(LOCAL_HEALTH_URL)
    if health and health.get("agent_initialized") is True:
        print(f"Server is already running and healthy at {LOCAL_HEALTH_URL}")
    else:
        print("A server is running but the agent is not initialized. Restarting with API key...")
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except Exception:
            SERVER_PROCESS.kill()
            SERVER_PROCESS.wait(timeout=5)
        is_running = False

if not is_running:
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not api_key:
        api_key = getpass("Enter Gemini API key for Web UI: ").strip()
        if not api_key:
            raise ValueError("Gemini API key is required to start analysis server.")

    server_env = os.environ.copy()
    server_env["GEMINI_API_KEY"] = api_key
    server_env["GOOGLE_API_KEY"] = api_key

    SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", HOST, "--port", str(PORT)],
        cwd=str(PROJECT_ROOT),
        env=server_env,
    )

    if not _wait_for_server(LOCAL_HEALTH_URL, timeout_seconds=30):
        raise RuntimeError("Server did not become ready in time. Check notebook output for errors.")

    health = _get_health(LOCAL_HEALTH_URL)
    if not health or health.get("agent_initialized") is not True:
        status = health.get("status") if isinstance(health, dict) else "unknown"
        detail = health.get("detail") if isinstance(health, dict) else "unknown"
        raise RuntimeError(
            f"Server is reachable but agent initialization failed (status={status}). Detail: {detail}"
        )

BASE_URL = LOCAL_HEALTH_URL
display(HTML(f'<a href="{BASE_URL}" target="_blank">Open Data Analysis Web UI (Local)</a>'))
print(f"Web UI ready: {BASE_URL}")
print("Local mode: open the URL above in your browser.")
print("Now upload/drop a CSV in the browser and click Analyze Dataset.")

RuntimeError: Server is reachable but agent initialization failed (status=degraded). Detail: Failed to initialize any Gemini model for this API key. Attempted: ['gemini-1.5-flash', 'gemini-2.0-flash', 'gemini-2.0-flash-lite']. Last error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
Please retry in 11.317033681s. [links {
  description: "Learn more about Gemini API quotas

/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/jasonhu/Documents/GitHub/CS539_Project/CS539_Project/src/agent.py:12: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
ImportError: _multiarray_umath failed to import
/opt/homebrew/Caskroom/miniforge/base/lib/python3.10/site-pack

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite
Please retry in 39.568087816s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash-lite"
  }
}
violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_i

INFO:     Started server process [54033]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [3]:
# Optional: stop Web UI server
STOP_SERVER_NOW = True

if STOP_SERVER_NOW and SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    SERVER_PROCESS.wait(timeout=10)
    print("Web UI server stopped.")
    
elif STOP_SERVER_NOW:
    print("No running Web UI server found.")
else:
    print("Stop skipped. Set STOP_SERVER_NOW = True to stop the server.")


Data Analysis Agent API - Shutting Down
Web UI server stopped.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [38094]
